# Option B — Landmark MLP (MediaPipe features, no pixels)
Runs MediaPipe HandLandmarker on ASL Alphabet training images offline,
extracts 21 3D joint positions, normalises them, trains a scikit-learn MLP.

**Why this works:** completely invariant to lighting, background, skin tone, camera quality.
MediaPipe sees the same 21 joints regardless of appearance.

Output: `models/optionB.pkl` — joblib bundle with mlp + scaler + classes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install & Imports

In [ ]:
!pip install -q kaggle mediapipe scikit-learn joblib

import os, random, joblib
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

print('mediapipe:', mp.__version__)

## 2. Download Dataset + Hand Model

In [ ]:
# ASL Alphabet real photos
if not os.path.exists('asl_alphabet_train'):
    os.environ['KAGGLE_USERNAME'] = 'abdullahashiry'
    os.environ['KAGGLE_KEY']      = 'KGAT_331632d901a6cb7a05431b55135bd8c2'
    !kaggle datasets download -d grassknoted/asl-alphabet --unzip

# MediaPipe hand landmarker
HAND_MODEL = 'hand_landmarker.task'
if not os.path.exists(HAND_MODEL):
    !wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
    print('Downloaded hand_landmarker.task')

TRAIN_DIR   = 'asl_alphabet_train/asl_alphabet_train'
ALL_CLASSES = sorted([c for c in os.listdir(TRAIN_DIR) if len(c) == 1 and c.isalpha()])
print(f'{len(ALL_CLASSES)} classes: {ALL_CLASSES}')

## 3. Extract Landmarks from All Images
MediaPipe processes each image and extracts 21 3D hand joint positions.
Images where no hand is detected are skipped.

In [ ]:
def landmarks_to_features(lm_list):
    """21 landmarks → 63-dim normalised vector. Invariant to scale/position."""
    pts = np.array([[lm.x, lm.y, lm.z] for lm in lm_list], dtype=np.float32)
    pts -= pts[0]                     # translate: wrist at origin
    scale = np.linalg.norm(pts[9])   # wrist→middle-finger-MCP distance
    if scale > 1e-6:
        pts /= scale
    return pts.flatten()              # (63,)


# Build hand detector (IMAGE mode for offline batch processing)
hand_options = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=HAND_MODEL),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.4,   # lower threshold — controlled photos
)
detector = mp_vision.HandLandmarker.create_from_options(hand_options)

X_feats, y_labels = [], []
skipped = 0

for cls in tqdm(ALL_CLASSES, desc='Classes'):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    cls_idx = ALL_CLASSES.index(cls)
    images  = [f for f in os.listdir(cls_dir)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    # Sample up to 2000 per class for speed (full = 3000)
    images  = random.sample(images, min(2000, len(images)))

    for fname in images:
        path = os.path.join(cls_dir, fname)
        try:
            mp_img  = mp.Image.create_from_file(path)
            result  = detector.detect(mp_img)
            if result.hand_landmarks:
                feats = landmarks_to_features(result.hand_landmarks[0])
                X_feats.append(feats)
                y_labels.append(cls_idx)
            else:
                skipped += 1
        except Exception:
            skipped += 1

detector.close()

X = np.array(X_feats, dtype=np.float32)
y = np.array(y_labels, dtype=np.int32)
print(f'\nExtracted: {len(X)} samples  Skipped (no hand): {skipped}')
print(f'Feature shape: {X.shape}   (21 joints × 3 axes = 63)')

## 4. Detection Rate per Class

In [ ]:
from collections import Counter
counts = Counter(y)
cls_names = [ALL_CLASSES[i] for i in sorted(counts)]
cls_counts = [counts[i] for i in sorted(counts)]
plt.bar(cls_names, cls_counts)
plt.xlabel('Letter'); plt.ylabel('Samples with detected hand')
plt.title('Landmark extraction success per class')
plt.tight_layout(); plt.show()

## 5. Train MLP Classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Training MLP...')
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,
    verbose=True,
)
mlp.fit(X_train, y_train)

y_pred  = mlp.predict(X_test)
acc     = accuracy_score(y_test, y_pred)
print(f'\nTest accuracy: {acc*100:.2f}%')

## 6. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=ALL_CLASSES))

# Loss curve
plt.plot(mlp.loss_curve_)
plt.xlabel('Iteration'); plt.ylabel('Loss')
plt.title('MLP training loss')
plt.tight_layout(); plt.show()

## 7. Save Model

In [ ]:
bundle = {
    'mlp':     mlp,
    'scaler':  scaler,
    'classes': np.array(ALL_CLASSES),   # index → letter string
}
joblib.dump(bundle, 'optionB.pkl')
print(f'Saved: optionB.pkl  ({os.path.getsize("optionB.pkl")/1e3:.0f} KB)')
print(f'Classes: {ALL_CLASSES}')

## 8. Copy to Drive

In [ ]:
import shutil

OUT = '/content/drive/MyDrive/CV552_SignLanguage/tflite'
os.makedirs(OUT, exist_ok=True)
shutil.copy('optionB.pkl', os.path.join(OUT, 'optionB.pkl'))
print(f'Saved to {OUT}')
print('\nNext: download optionB.pkl → bonus/webcam_trials/models/optionB.pkl')